# Notebook 10 — Model Interpretability

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 10 of 12  
**Time estimate:** 75 minutes

> A model that predicts correctly but cannot be interpreted is not useful for
> biology — we need to know which genomic positions, which residues, which
> features matter. Saliency maps and integrated gradients are the standard
> attribution methods for sequence models in genomics.

---
## Step 1 — Motivation

You have a CNN that predicts TF binding with AUROC=0.95. Your collaborator asks:
"Which positions in the sequence drive the prediction?" Saliency maps compute
$\partial \hat{y} / \partial x_i$ — how sensitive is the output to each input position.
Integrated gradients accumulate these sensitivities along a path from a reference
to the actual input, satisfying the axiom that the sum of attributions equals
the prediction difference from baseline.

---
## Step 2 — Intuition

**Saliency map (vanilla gradient):**
For input $x$ and output $\hat{y}(x)$:
$$S_i = \frac{\partial \hat{y}}{\partial x_i}$$
Compute via one backward pass. Fast. But noisy — gradient can be high
in regions where the model is locally sensitive but globally irrelevant.

**Integrated gradients (Sundararajan 2017):**
Accumulate gradients along a straight-line path from a reference (baseline $x'$, e.g., all zeros)
to the actual input $x$:
$$\text{IG}_i(x) = (x_i - x'_i) \times \int_0^1 \frac{\partial \hat{y}(x' + \alpha(x-x'))}{\partial x_i}\, d\alpha$$

Approximated with $m$ steps:
$$\text{IG}_i(x) \approx (x_i - x'_i) \times \frac{1}{m} \sum_{k=1}^m \frac{\partial \hat{y}(x' + \frac{k}{m}(x-x'))}{\partial x_i}$$

**Completeness axiom:** $\sum_i \text{IG}_i(x) = \hat{y}(x) - \hat{y}(x')$.
The attributions sum to the prediction difference from baseline.

**GradCAM (for CNNs):**
Weight spatial feature maps by the gradient of the class score w.r.t.
those feature maps → activation heatmap. Simpler than IG, works well for CNNs.

**Attention as attribution:**
In transformers, attention weights are often used as a proxy for importance.
Caution: attention weights ≠ attribution in general (high attention ≠ high importance).
Use gradient-weighted attention for more reliable attribution.

---
## Step 3 — Biological Background

**Interpretation of sequence attribution in genomics:**
- If a CNN is trained on ChIP-seq peaks, the positions with highest attribution
  should correspond to the TF binding motif
- Validate: extract the highest-attribution subsequences from many positive
  sequences → cluster into a PWM → compare to JASPAR
- This is how DeepBind validates its learned filters

**TF-MoDISco (Shrikumar 2018, Bioinformatics):**
Applies integrated gradients to ChIP-seq models, then clusters the resulting
attribution patterns to discover de novo TF binding motifs.
Standard tool for interpreting trained genomics models.

**Enformer attribution:**
For Enformer, attribution across a 196kb window shows which specific enhancer
regions drive predictions for specific genes — directly identifying regulatory elements.

**In silico mutagenesis:**
Simpler but expensive approach: create all single-nucleotide variants of the input
sequence, compute the prediction for each, subtract the wildtype prediction.
The position with the largest drop in binding score (when mutated) is the most important.
This is computationally expensive (4L forward passes per sequence) but
requires no gradient computation — works with any black-box model.

---
## Step 4 — Mathematical Explanation

**Integrated gradients formal statement:**

For a function $F: \mathbb{R}^n \rightarrow [0,1]$ (the model), baseline $x' \in \mathbb{R}^n$:
$$\text{IG}_i(x) = (x_i - x'_i) \int_0^1 \frac{\partial F(x' + \alpha(x-x'))}{\partial x_i} d\alpha$$

**Completeness:** $\sum_{i=1}^n \text{IG}_i(x) = F(x) - F(x')$
(follows from the fundamental theorem of calculus applied to $F$ along the path).

**Sensitivity:** If $F(x) \neq F(x')$ and $F$ depends on feature $i$, then $\text{IG}_i(x) \neq 0$.

**Dummy:** If $F$ does not depend on feature $i$, then $\text{IG}_i(x) = 0$.

**Baseline choice:**
For one-hot encoded DNA: baseline = all-zeros tensor (no nucleotide information).
For expression data: baseline = mean expression across samples.
The baseline represents "no signal" — the attribution measures departure from this.

In [ ]:
# Step 6 — Python: Saliency maps + Integrated Gradients for CNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Re-use NB03 CNN and dataset ----
NUCL = 'ACGT'; NUCL_IDX = {b: i for i, b in enumerate(NUCL)}
MOTIF = 'CCGCGNGGNGG'; MOTIF_L = 11; SEQ_L = 50
rng = np.random.default_rng(42)

def generate_seqs(n=1000):
    seqs, labels, motif_positions = [], [], []
    for i in range(n):
        seq = list(''.join(rng.choice(list(NUCL), SEQ_L)))
        if i < n//2:
            pos = rng.integers(5, SEQ_L-MOTIF_L-5)
            for j, b in enumerate(MOTIF):
                if b != 'N': seq[pos+j] = b if rng.random()>0.1 else rng.choice(list(NUCL))
            labels.append(1); motif_positions.append(pos)
        else:
            labels.append(0); motif_positions.append(-1)
        seqs.append(''.join(seq))
    return seqs, np.array(labels), np.array(motif_positions)

def one_hot_batch(sequences):
    X = np.zeros((len(sequences), 4, SEQ_L), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j, b in enumerate(seq):
            if b in NUCL_IDX: X[i, NUCL_IDX[b], j] = 1.0
    return X

seqs, y_seq, motif_pos = generate_seqs(1000)
X_oh = one_hot_batch(seqs)

class DeepBind(nn.Module):
    def __init__(self, n_filters=16, kernel_size=11, fc_units=32):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc1 = nn.Linear(n_filters, fc_units)
        self.fc2 = nn.Linear(fc_units, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        h = self.pool(self.relu(self.conv(x))).squeeze(-1)
        return self.sigmoid(self.fc2(self.relu(self.fc1(h))))

# Train quickly
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X); self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

full_ds = SeqDS(X_oh, y_seq)
train_ds, val_ds = torch.utils.data.random_split(full_ds, [800, 200])
train_ld = DataLoader(train_ds, batch_size=64, shuffle=True)

model = DeepBind(16, 11, 32).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(30):
    model.train()
    for X_b, y_b in train_ld:
        optimizer.zero_grad()
        loss = nn.BCELoss()(model(X_b.to(device)), y_b.to(device))
        loss.backward(); optimizer.step()
print('Model trained')

# ---- Saliency map ----
def saliency_map(model, x):
    """Vanilla gradient saliency: d(output) / d(input)."""
    x = x.clone().requires_grad_(True)
    output = model(x.unsqueeze(0))
    output.backward()
    return x.grad.detach()  # (4, L)

# ---- Integrated gradients ----
def integrated_gradients(model, x, baseline=None, n_steps=50):
    """
    Compute IG for a single input x (shape: 4 x L).
    baseline: same shape as x, default = zeros
    Returns: attribution tensor (4, L)
    """
    if baseline is None:
        baseline = torch.zeros_like(x)
    
    # Interpolate between baseline and input
    alphas = torch.linspace(0, 1, n_steps)
    interpolated = torch.stack([baseline + a*(x-baseline) for a in alphas])  # (n_steps, 4, L)
    interpolated.requires_grad_(True)
    
    # Batch forward pass
    outputs = model(interpolated)  # (n_steps, 1)
    outputs.sum().backward()
    
    # Average gradients
    avg_grads = interpolated.grad.mean(dim=0)  # (4, L)
    
    # Multiply by (x - baseline)
    ig = (x - baseline) * avg_grads
    return ig.detach()

# ---- Compute attributions for positive sequences ----
model.eval()
pos_idx = np.where(y_seq == 1)[0][:20]
neg_idx = np.where(y_seq == 0)[0][:20]

ig_pos = []
sal_pos = []
for i in pos_idx:
    x = torch.tensor(X_oh[i]).to(device)
    ig = integrated_gradients(model, x)
    ig_pos.append(ig.cpu().numpy().sum(axis=0))  # sum over nucleotides → per-position score
    sal = saliency_map(model, x)
    sal_pos.append(sal.abs().cpu().numpy().sum(axis=0))

ig_pos = np.array(ig_pos)  # (20, L)
sal_pos = np.array(sal_pos)

# Also compute for negative sequences
ig_neg = []
for i in neg_idx:
    x = torch.tensor(X_oh[i]).to(device)
    ig = integrated_gradients(model, x)
    ig_neg.append(ig.cpu().numpy().sum(axis=0))
ig_neg = np.array(ig_neg)

print(f'IG computed for {len(ig_pos)} positive and {len(ig_neg)} negative sequences')
print(f'Completeness check (should be close to pred - baseline_pred):')
x_test = torch.tensor(X_oh[pos_idx[0]]).to(device)
ig_test = integrated_gradients(model, x_test)
with torch.no_grad():
    pred = model(x_test.unsqueeze(0)).item()
    base_pred = model(torch.zeros_like(x_test).unsqueeze(0)).item()
print(f'  IG sum: {ig_test.sum().item():.4f}  |  pred - baseline: {pred - base_pred:.4f}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: Mean IG per position (positive sequences)
ax = axes[0]
mean_ig_pos = ig_pos.mean(axis=0)  # (L,)
mean_ig_neg = ig_neg.mean(axis=0)
ax.plot(mean_ig_pos, 'tomato', lw=2, label='Positive (bound)')
ax.plot(mean_ig_neg, 'steelblue', lw=1.5, alpha=0.7, label='Negative (unbound)')
ax.set_xlabel('Sequence position')
ax.set_ylabel('Mean integrated gradient')
ax.set_title('A. Mean IG per position\n(positive seqs should peak at motif)')
ax.legend(fontsize=9)

# Panel B: IG heatmap for 20 positive sequences (sorted by motif position)
ax = axes[1]
sorted_ig = ig_pos  # already in order
im = ax.imshow(sorted_ig, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Sequence position'); ax.set_ylabel('Positive sequence')
ax.set_title('B. IG heatmap (20 positive seqs)\n(bright = attributable to binding prediction)')
# Mark approximate motif positions
for row_i, seq_i in enumerate(pos_idx):
    mp = motif_pos[seq_i]
    if mp >= 0 and mp < SEQ_L:
        ax.add_patch(plt.Rectangle((mp-0.5, row_i-0.5), MOTIF_L, 1,
                                      fill=False, edgecolor='yellow', lw=1.5))

# Panel C: IG vs. saliency comparison (one positive sequence)
ax = axes[2]
example_idx = pos_idx[0]
ig_one = ig_pos[0]
sal_one = sal_pos[0]
ax.plot(ig_one, 'tomato', lw=2, label='Integrated Gradients')
ax.plot(sal_one, 'steelblue', lw=1.5, alpha=0.8, label='Saliency (|grad|)')
mp = motif_pos[example_idx]
if mp >= 0:
    ax.axvspan(mp, mp+MOTIF_L, alpha=0.2, color='green', label=f'True motif ({mp}-{mp+MOTIF_L})')
ax.set_xlabel('Sequence position')
ax.set_ylabel('Attribution score')
ax.set_title('C. IG vs. saliency (one sequence)\n(IG is smoother, satisfies completeness)')
ax.legend(fontsize=7)

# Panel D: In silico mutagenesis for one sequence
ax = axes[3]
model.eval()
x0 = torch.tensor(X_oh[example_idx]).to(device)
with torch.no_grad():
    wt_score = model(x0.unsqueeze(0)).item()

ism_scores = np.zeros((4, SEQ_L))  # mutation effect per position and nucleotide
with torch.no_grad():
    for pos in range(SEQ_L):
        for nuc in range(4):
            x_mut = x0.clone()
            x_mut[:, pos] = 0
            x_mut[nuc, pos] = 1
            mut_score = model(x_mut.unsqueeze(0)).item()
            ism_scores[nuc, pos] = wt_score - mut_score  # positive = mutation hurts binding

im = ax.imshow(ism_scores, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_yticks(range(4)); ax.set_yticklabels(list(NUCL))
ax.set_xlabel('Position'); ax.set_ylabel('Nucleotide')
ax.set_title('D. In silico mutagenesis\n(red = mutation hurts binding; blue = neutral)')
if mp >= 0: ax.axvspan(mp-0.5, mp+MOTIF_L-0.5, alpha=0.15, color='green')

plt.suptitle('Module 14 NB10: Model Interpretability', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('model_interpretability.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement integrated gradients from scratch and verify the completeness axiom:
   $\sum_i \text{IG}_i(x) \approx \hat{y}(x) - \hat{y}(x')$. How close do you get
   with $m=50$ steps vs. $m=200$ steps?
2. Apply the attribution scores to extract motifs: take the top-10 positions per
   positive sequence (by IG magnitude), extract the subsequence, compute a PWM.
   Compare to the planted MOTIF.
3. Compare vanilla saliency, saliency × input (elementwise), and integrated gradients
   for the same sequence. Which recovers the motif region most cleanly?
4. Apply in silico mutagenesis to all 20 positive sequences. Average the mutation
   effects to get a consensus pattern. Does it match the planted motif better than IG?

---
## Step 10 — Quiz

1. Write the integrated gradients formula. What is the completeness axiom?
2. What is the baseline $x'$ and how should it be chosen for one-hot DNA sequences?
3. What is the difference between vanilla saliency and integrated gradients?
   When does saliency fail (saturation problem)?
4. What is in silico mutagenesis? What are its advantages and disadvantages
   compared to gradient-based attribution?
5. Why should attention weights NOT be used directly as attribution scores in transformers?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the completeness axiom of integrated gradients
> makes it more trustworthy than vanilla saliency for scientific reporting,
> and describe how you would validate the attributions against a known biological
> ground truth (e.g., a JASPAR motif).]*

---
*Next: `11_alphafold_and_language_models.ipynb`*